# Content-based book recommendations

This notebook compares books using text features and cosine similarity. It demonstrates a full book-to-book similarity matrix followed by a separate title-query example. No user ratings or reading history are used, so this is a content-based approach rather than collaborative filtering.

## Run locally or in Google Colab

Run this notebook locally from the repository root or the `notebooks` directory. It uses `datasets/BX-Books.csv`. If needed, install dependencies in a notebook cell with `%pip install pandas scikit-learn`.

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AvansICT/SE-2.1-2.2-LU2/blob/main/notebooks/06_Recommender%20systems.ipynb)

If local execution is slow, you can try Colab using the badge. It opens the version saved on GitHub; local edits become available there after they are pushed. To run your current local version immediately, upload this notebook to Colab instead. The loading cell automatically downloads the dataset from this repository when running in Colab, without Google Drive. If the download is unavailable, it asks you to upload `BX-Books.csv` from your local `datasets` directory. Colab is not guaranteed to be faster; these scikit-learn operations run on the CPU and do not use a GPU.

## 1. Import libraries

Pandas handles book records, `CountVectorizer` converts text into term-count vectors, and cosine similarity compares those vectors.

In [1]:
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity


## 2. Load the book metadata

The code locates `datasets/BX-Books.csv` relative to the repository root or the `notebooks` directory. In Colab, it downloads the file from GitHub if it is not already present. Both CSV-loading examples use the same `data_path`.

The file has a header with ISBN, title, author, publication year, publisher, and image URLs. Each row represents a book record. The CSV is read using semicolons and Latin-1 encoding; `on_bad_lines="skip"` skips malformed records rather than repairing them.


In [2]:
from pathlib import Path
import sys
from urllib.request import urlretrieve
from urllib.error import HTTPError, URLError

# Support execution from either the repository root or notebooks directory.
data_path = next(
    (path for path in (Path('datasets/BX-Books.csv'), Path('../datasets/BX-Books.csv'))
     if path.is_file()),
    None,
)

if data_path is None:
    if 'google.colab' in sys.modules:
        data_path = Path('/content/datasets/BX-Books.csv')
        data_path.parent.mkdir(parents=True, exist_ok=True)
        try:
            urlretrieve(
                'https://raw.githubusercontent.com/AvansICT/SE-2.1-2.2-LU2/main/datasets/BX-Books.csv',
                data_path,
            )
        except (HTTPError, URLError):
            from google.colab import files

            print('Upload BX-Books.csv from your local datasets directory.')
            uploaded = files.upload()
            if 'BX-Books.csv' not in uploaded:
                raise FileNotFoundError('Please upload the file named BX-Books.csv.')
            data_path.write_bytes(uploaded['BX-Books.csv'])
    else:
        raise FileNotFoundError(
            'Cannot find datasets/BX-Books.csv. Run from the repository root '
            'or the notebooks directory.'
        )

print(f'Loading dataset: {data_path.resolve()}')
df = pd.read_csv(data_path, on_bad_lines='skip', encoding='latin-1', sep=';')
df.head()


Loading dataset: /Users/nannalakens/School/Jaar2/P2.1-2.2/Lu2/SE-2.1-2.2-LU2/datasets/BX-Books.csv


/var/folders/64/sqwyjtfn7cv4kj35pqkzbg300000gn/T/ipykernel_10295/24420165.py:37: DtypeWarning: Columns (3: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, on_bad_lines='skip', encoding='latin-1', sep=';')


,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


### Inspect the loaded data

`df.info()` shows column names, data types, and non-null counts. Check that the expected columns were loaded and look for missing values before constructing text features.

In [3]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  str   
 1   Book-Title           271360 non-null  str   
 2   Book-Author          271358 non-null  str   
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  str   
 5   Image-URL-S          271360 non-null  str   
 6   Image-URL-M          271360 non-null  str   
 7   Image-URL-L          271357 non-null  str   
dtypes: object(1), str(7)
memory usage: 16.6+ MB


### Optional preprocessing examples

The following commented cells show alternative CSV loading, text cleaning, and export operations. They do not execute as written. The alternative loader assumes a file without a header, so check the actual file format before using it. The export example would overwrite the CSV at the specified path if enabled.

In [51]:
# import pandas as pd

# column_names = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
#                 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

# # Read CSV safely
# df = pd.read_csv(
#     data_path,
#     sep=';',
#     encoding='latin-1',
#     header=None,
#     names=column_names,
#     engine='python',
#     quotechar='"',
#     on_bad_lines='skip'  # skip problematic lines
# )



In [52]:
# import pandas as pd

# # Noms de colonnes souhaités
# column_names = ['ISBN', 'Book-Title', 'Book-Author', 'Year-Of-Publication',
#                 'Publisher', 'Image-URL-S', 'Image-URL-M', 'Image-URL-L']

# # Lire le CSV
# df = pd.read_csv(
#     data_path,
#     sep=';',
#     encoding='latin-1',
#     header=None,
#     names=column_names,
#     engine='python',
#     quotechar='"',
#     on_bad_lines='skip'
# )


In [53]:
# # Exemple : minuscule et nettoyage des espaces
# df['Book-Title'] = df['Book-Title'].str.lower().str.strip()
# df['Book-Author'] = df['Book-Author'].str.lower().str.replace(' ', '')
# df['Publisher'] = df['Publisher'].str.lower().str.strip()


In [54]:
# df.to_csv(data_path,
#           sep=';',           # même séparateur que l'original
#           encoding='latin-1',
#           index=False)       # ne pas écrire l'index Pandas


## 3. Remove duplicate titles

Preview the records and count repeated `Book-Title` values. `drop_duplicates` keeps the first record for each exact title, then the second count checks the result. Books with the same title may have different authors or editions; title-based deduplication removes those distinctions. This step runs before lowercasing, so capitalization variants may remain.

In [4]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 271360 entries, 0 to 271359
Data columns (total 8 columns):
 #   Column               Non-Null Count   Dtype 
---  ------               --------------   ----- 
 0   ISBN                 271360 non-null  str   
 1   Book-Title           271360 non-null  str   
 2   Book-Author          271358 non-null  str   
 3   Year-Of-Publication  271360 non-null  object
 4   Publisher            271358 non-null  str   
 5   Image-URL-S          271360 non-null  str   
 6   Image-URL-M          271360 non-null  str   
 7   Image-URL-L          271357 non-null  str   
dtypes: object(1), str(7)
memory usage: 16.6+ MB


In [6]:
df.duplicated(subset='Book-Title').sum()

np.int64(29225)

In [7]:
df = df.drop_duplicates(subset='Book-Title')

In [8]:
df.duplicated(subset='Book-Title').sum()

np.int64(0)

## 4. Select a reproducible sample

The default sample contains up to 3,000 books to keep local execution manageable. The fixed random state makes the selection reproducible for the same input data. Resetting the index aligns row positions with the similarity matrix created later.

The dense pairwise matrix needs about 72 MB for 3,000 books, excluding other objects and copies. You can increase `sample_size` to 15,000 if enough memory is available, but that matrix alone needs about 1.8 GB. Colab is an optional place to try a larger sample.


In [9]:
sample_size = 3000
df = df.sample(n=min(sample_size, len(df)), replace=False, random_state=490)
df = df.reset_index(drop=True)


In [10]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0060972491,"Thank You, Jeeves",P.G. Wodehouse,1989,Perennial,http://images.amazon.com/images/P/0060972491.0...,http://images.amazon.com/images/P/0060972491.0...,http://images.amazon.com/images/P/0060972491.0...
1,015602781X,Pincher Martin: The Two Deaths of Christopher ...,William Golding,2002,Harvest Books,http://images.amazon.com/images/P/015602781X.0...,http://images.amazon.com/images/P/015602781X.0...,http://images.amazon.com/images/P/015602781X.0...
2,1561441627,The Nutcracker (Christmas Fun-to-Read Fairy Ta...,Roberta Collier-Morales,1992,Modern Publishing,http://images.amazon.com/images/P/1561441627.0...,http://images.amazon.com/images/P/1561441627.0...,http://images.amazon.com/images/P/1561441627.0...
3,1402708610,I Know a Rhino,Charles Fuge,2003,Sterling,http://images.amazon.com/images/P/1402708610.0...,http://images.amazon.com/images/P/1402708610.0...,http://images.amazon.com/images/P/1402708610.0...
4,0441586325,Norby and the Oldest Dragon,Janet Asimov,1993,Ace Books,http://images.amazon.com/images/P/0441586325.0...,http://images.amazon.com/images/P/0441586325.0...,http://images.amazon.com/images/P/0441586325.0...


## 5. Normalize authors, titles, and publishers

`clean_text` lowercases author names and removes spaces, so a multiword name becomes a single token. The subsequent cells lowercase titles and publishers. These transformations reduce formatting differences; they do not resolve spelling variants or identify authors uniquely. Converting a missing author with `str` can also introduce the literal token `nan`.

In [11]:
def clean_text(author):
    result = str(author).lower()
    return(result.replace(' ',''))

In [12]:
df['Book-Author'] = df['Book-Author'].apply(clean_text)

In [13]:
df.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0060972491,"Thank You, Jeeves",p.g.wodehouse,1989,Perennial,http://images.amazon.com/images/P/0060972491.0...,http://images.amazon.com/images/P/0060972491.0...,http://images.amazon.com/images/P/0060972491.0...
1,015602781X,Pincher Martin: The Two Deaths of Christopher ...,williamgolding,2002,Harvest Books,http://images.amazon.com/images/P/015602781X.0...,http://images.amazon.com/images/P/015602781X.0...,http://images.amazon.com/images/P/015602781X.0...
2,1561441627,The Nutcracker (Christmas Fun-to-Read Fairy Ta...,robertacollier-morales,1992,Modern Publishing,http://images.amazon.com/images/P/1561441627.0...,http://images.amazon.com/images/P/1561441627.0...,http://images.amazon.com/images/P/1561441627.0...
3,1402708610,I Know a Rhino,charlesfuge,2003,Sterling,http://images.amazon.com/images/P/1402708610.0...,http://images.amazon.com/images/P/1402708610.0...,http://images.amazon.com/images/P/1402708610.0...
4,0441586325,Norby and the Oldest Dragon,janetasimov,1993,Ace Books,http://images.amazon.com/images/P/0441586325.0...,http://images.amazon.com/images/P/0441586325.0...,http://images.amazon.com/images/P/0441586325.0...


In [14]:
df['Book-Title'] = df['Book-Title'].str.lower()
df['Publisher'] = df['Publisher'].str.lower()

## 6. Construct text features

The code removes identifiers, image URLs, and publication year, then joins the remaining columns starting at position 1 into `data`. With the expected order of title, author, and publisher, this combines **author and publisher**, excluding the title. Inspect the printed text to confirm which fields contribute to similarity.

In [15]:
# combine all strings:
df2 = df.drop(['ISBN','Image-URL-S','Image-URL-M','Image-URL-L','Year-Of-Publication'],axis=1)

df2['data'] = df2[df2.columns[1:]].apply(
    lambda x: ' '.join(x.dropna().astype(str)),
    axis=1
)
print(df2['data'].head())

0                     p.g.wodehouse perennial
1                williamgolding harvest books
2    robertacollier-morales modern publishing
3                        charlesfuge sterling
4                       janetasimov ace books
Name: data, dtype: str


### Convert metadata into counts

`CountVectorizer` creates a sparse matrix with one row per book and one column per vocabulary term. Each value counts a term in that book?s combined metadata. Shared tokens contribute to similarity; the representation does not understand a book?s plot or meaning.

In [16]:
from sklearn.feature_extraction.text import CountVectorizer

vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df2['data'])

## 7. Compare every pair of books

Cosine similarity compares vector directions. For these nonnegative count vectors, scores range from 0 (no shared terms) to 1 (identical directions). The result is a square matrix: entry `(i, j)` compares books at positions `i` and `j`. Nonzero vectors have self-similarity 1; empty vectors do not.

The full matrix is dense even though the input counts are sparse.

In [17]:
from sklearn.metrics.pairwise import cosine_similarity

similarities = cosine_similarity(vectorized)

In [18]:
print(similarities)

[[1. 0. 0. ... 0. 0. 0.]
 [0. 1. 0. ... 0. 0. 0.]
 [0. 0. 1. ... 0. 0. 0.]
 ...
 [0. 0. 0. ... 1. 0. 0.]
 [0. 0. 0. ... 0. 1. 0.]
 [0. 0. 0. ... 0. 0. 1.]]


### Label the similarity matrix

Titles become row and column labels, making the scores easier to inspect. This assignment replaces `df` with the similarity table, so it no longer holds the original metadata. This section builds the table but does not yet select recommendations. For book-to-book recommendations, exclude the selected book itself when ranking scores.

In [19]:
df = pd.DataFrame(similarities, columns=df['Book-Title'], index=df['Book-Title']).reset_index()

df.head()

Book-Title,Book-Title,"thank you, jeeves",pincher martin: the two deaths of christopher martin,the nutcracker (christmas fun-to-read fairy tales ser),i know a rhino,norby and the oldest dragon,his eminence and hizzoner,el oro de los sueã±os (leer en espaã±ol nivel 4),celestine insights,feng shui and money: a nine-week program for creating wealth using ancient principles and techniques,...,whisper his name,sprout mask replica,flounder,knowing where to look: the ultimate guide to research,"a surfeit of guns (missing mystery, 20)","the cat who would be president: or, how i clawed my way to the top",law school confidential : a complete guide to the law school experience,mystery in washington (boxcar children special (paperback)),success through a positive mental attitude,the devil wears prada
0,"thank you, jeeves",1.0,0.000000,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
1,pincher martin: the two deaths of christopher ...,0.0,1.000000,0.0,0.0,0.333333,0.288675,0.000000,0.288675,0.0,...,0.333333,0.0,0.333333,0.288675,0.0,0.0,0.0,0.0,0.0,0.0
2,the nutcracker (christmas fun-to-read fairy ta...,0.0,0.000000,1.0,0.0,0.000000,0.000000,0.223607,0.000000,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
3,i know a rhino,0.0,0.000000,0.0,1.0,0.000000,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0,0.0,0.0
4,norby and the oldest dragon,0.0,0.333333,0.0,0.0,1.000000,0.288675,0.000000,0.288675,0.0,...,0.333333,0.0,0.333333,0.288675,0.0,0.0,0.0,0.0,0.0,0.0


In [20]:
print(df.columns)   # lst the column names


Index(['Book-Title', 'thank you, jeeves',
       'pincher martin: the two deaths of christopher martin',
       'the nutcracker (christmas fun-to-read fairy tales ser)',
       'i know a rhino', 'norby and the oldest dragon',
       'his eminence and hizzoner',
       'el oro de los sueã±os (leer en espaã±ol nivel 4)',
       'celestine insights',
       'feng shui and money: a nine-week program for creating wealth using ancient principles and techniques',
       ...
       'whisper his name', 'sprout mask replica', 'flounder',
       'knowing where to look: the ultimate guide to research',
       'a surfeit of guns (missing mystery, 20)',
       'the cat who would be president: or, how i clawed my way to the top',
       'law school confidential : a complete guide to the law school experience',
       'mystery in washington (boxcar children special (paperback))',
       'success through a positive mental attitude', 'the devil wears prada'],
      dtype='str', name='Book-Title', length

## 8. Retrieve books from a title query

This example reloads the CSV using `data_path` from section 2 and rebuilds text features from **title, author, and publisher**. It does not reuse the earlier sample or deduplication. The CSV header supplies the column names.

The input title is transformed using the fitted vocabulary and compared with every book. Only one query-to-catalog score vector is calculated, avoiding the full pairwise matrix. Scores are sorted in descending order and the ten highest are displayed.

This is text matching against a typed title, rather than a lookup of the selected book?s complete metadata. An exact match can appear in the results, duplicates are possible, and a query containing only unknown terms produces zero scores.

### Interpret the results

A high score indicates overlapping text features, not a probability that someone will enjoy the book. Inspect whether the results are useful and varied. No held-out relevance judgments or user feedback are evaluated in this notebook.

In [21]:
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd

# Reload the local dataset, using its existing header.
df = pd.read_csv(data_path, sep=';', encoding='latin-1', on_bad_lines='skip')

# Cleaning
df['Book-Title'] = df['Book-Title'].str.lower().str.strip()
df['Book-Author'] = df['Book-Author'].str.lower().str.replace(' ', '')
df['Publisher'] = df['Publisher'].str.lower().str.strip()

# Combine columns
df['data'] = df[['Book-Title', 'Book-Author', 'Publisher']].apply(lambda x: ' '.join(x.dropna()), axis=1)

# Vectorization
vectorizer = CountVectorizer()
vectorized = vectorizer.fit_transform(df['data'])

# Input book
input_book = 'far beyond the stars (star trek deep space nine)'
input_processed = input_book.lower().strip()

# Vectorize the input
input_vector = vectorizer.transform([input_processed])

# Compute cosine similarity
similarities = cosine_similarity(input_vector, vectorized).flatten()

# Top 10 books with similarity score
top_indices = similarities.argsort()[::-1][0:10]
top_books = df.iloc[top_indices][['Book-Title']].copy()
top_books['similarity_score'] = similarities[top_indices]

print(top_books)


/var/folders/64/sqwyjtfn7cv4kj35pqkzbg300000gn/T/ipykernel_10295/2698260479.py:6: DtypeWarning: Columns (3: Year-Of-Publication) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(data_path, sep=';', encoding='latin-1', on_bad_lines='skip')


                                               Book-Title  similarity_score
189472   far beyond the stars (star trek deep space nine)          0.889297
155240  the deep space nine technical manual (star tre...          0.748455
23501              the search (star trek deep space nine)          0.712697
240725  deep space nine companion (star trek deep spac...          0.695048
117013  star trek deep space nine warped (star trek de...          0.695048
36261         the siege (star trek deep space nine, no 2)          0.688530
62934   the way of the warrior (star trek deep space n...          0.688247
95933   worlds of deep space nine, vol i (star trek de...          0.680414
123636        the star ghost (star trek: deep space nine)          0.673575
96045   the mist:  the captain's table book 3 (star tr...          0.670820
